In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


def load_data(file_path):
    df = pd.read_csv(file_path) 
    df = df.dropna(subset=['text', 'label'])
    return df

df = load_data('NLP_project.csv')
df["label"] = df["label"].map({
    "supportive": 1,
    "unsupportive": 0
})
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(df.head())


   id                                               text  label
0   1  Miss Potwine gave into my keeping your preciou...      1
1   2  the value of health to a lady is inestimable. ...      1
2   3  Yet despite the emphasis on religion at Holyok...      1
3   4  Pupils often remark a decided improvement in t...      1
4   5  It is a duty to be healthy. The rules of this ...      1


In [2]:
class LogisticRegression:
            
    def __init__(self):
        self.weight = None
        self.bias = 0 
        self.mean = None
        self.std = None

    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))

    def loss(self, a, y):
        return (-1/y.shape[0])*(np.dot(y.T,(np.log(a))) + np.dot((1-y).T,(np.log(1-a))))
    
    def train(self, X, y, alpha = 0.1, numIters=100):
        X = np.array(X)
        self.Y = np.array(y).reshape((-1, 1))
        
        # scaling
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1
        self.X_scaled = (X - self.mean) / self.std

        # initial weights
        n_features = self.X_scaled.shape[1]
        self.weight = np.zeros((n_features, 1))
        self.bias = 0
        self.gradientDescent(alpha=alpha, numIters=numIters)

    def gradientDescent(self, alpha, numIters):
        # gradient descent
        loss = 0
        for i in range(numIters):
            # dot product of feature and weight plus bias
            Z = np.dot(self.X_scaled, self.weight) + self.bias
            A = self.sigmoid(Z)
            
            # gradients
            grad = np.dot(self.X_scaled.T, (A - self.Y)) / self.Y.shape[0]
            db = np.sum(A - self.Y) / self.Y.shape[0]

            prevLoss = loss
            loss = self.loss(A, self.Y)
            stepSize = abs(prevLoss - loss)

            # update
            self.weight -= alpha * grad
            self.bias -= alpha * db

            if stepSize.any() < 0.000001:
                break

            

    def predict(self, X):
        X_test_scaled = (np.array(X) - self.mean) / self.std
        z = np.dot(X_test_scaled, self.weight) + self.bias
        y_pred = self.sigmoid(z)
        return (y_pred >= 0.5).astype(int)

In [ ]:
# B.manually define features
def extract_manual_features(text):
    text = text.lower()

    # word lists
    pos_words = ["exercise", "gymnasium", "athletic", "health", "games"]
    neg_words = ["weak", "delicate", "strain", "masculine", "fragile"]

    # features
    pos_count = sum(word in text for word in pos_words)
    neg_count = sum(word in text for word in neg_words)

    has_modal = int("should" in text or "must" in text)
    has_negation = int("not" in text or "no" in text)

    pronouns = ["she", "her", "woman", "women", "girl"]
    pronoun_count = sum(word in text for word in pronouns)

    length = len(text.split())

    return np.array([
        pos_count,
        neg_count,
        has_modal,
        has_negation,
        pronoun_count,
        length
    ], dtype=float)

In [4]:
# label
Y_train = train_df['label'].values
Y_test = test_df['label'].values

# A.automatically extract features
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')
X_train_auto = vectorizer.fit_transform(train_df['text']).toarray() 
X_test_auto = vectorizer.transform(test_df['text']).toarray()

X_train_manual = [extract_manual_features(t) for t in train_df['text']]
X_test_manual = [extract_manual_features(t) for t in test_df['text']]

In [5]:
# ===auto===
model_auto = LogisticRegression()
model_auto.train(X_train_auto, Y_train, alpha=0.1, numIters=100)

# ===manual===
model_manual = LogisticRegression()
model_manual.train(X_train_manual, Y_train, alpha=0.1, numIters=100)

In [6]:
def evaluate (model, X, Y, name = 'model'):
    y_pred = model.predict(X)
    accuracy = np.mean(y_pred.flatten() == Y.flatten())
    print(name)
    print('Accuracy: {}'.format(accuracy))

evaluate(model_auto, X_test_auto, Y_test, name = "TF-IDF")
evaluate(model_manual, X_test_manual, Y_test, name = "manual features")

TF-IDF
Accuracy: 0.5714285714285714
manual features
Accuracy: 0.7142857142857143


In [7]:
feature_names = [
    "pos_count",
    "neg_count",
    "had_modal",
    "negation",
    "pronouns",
    "length"
]

coefficients = model_manual.weight.flatten()

print("Feature Weights:")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")

Feature Weights:
pos_count: 1.2735
neg_count: -0.2959
had_modal: -0.2768
negation: 0.4300
pronouns: 0.4733
length: 0.9001
